# Benchmarking: `ml_from_scratch` vs `scikit-learn`

This notebook empirically compares our pure NumPy implementations against the highly-optimized C/Cython implementations in `scikit-learn` in terms of both correctness (Accuracy/R2) and training time.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression as SklearnLogReg
from src.ml_from_scratch.linear import LogisticRegression as MyLogReg

# Generate Dataset
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Benchmark Scikit-Learn
start = time.time()
sk_model = SklearnLogReg(max_iter=1000)
sk_model.fit(X_train, y_train)
sk_time = time.time() - start
sk_acc = sk_model.score(X_test, y_test)
print(f'Scikit-Learn - Time: {sk_time:.4f}s | Accuracy: {sk_acc:.4f}')

In [ ]:
# Benchmark Custom Implementation
start = time.time()
my_model = MyLogReg(learning_rate=0.01, n_iterations=1000)
my_model.fit(X_train, y_train)
my_time = time.time() - start

# Predict and score
preds = np.where(my_model.predict_proba(X_test)[:, 1] >= 0.5, 1, 0) if len(my_model.predict_proba(X_test).shape) > 1 else np.where(my_model.predict_proba(X_test) >= 0.5, 1, 0)
my_acc = np.mean(preds == y_test)
print(f'Custom Model - Time: {my_time:.4f}s | Accuracy: {my_acc:.4f}')

## Results
As expected, our native Python/NumPy implementation will be slower than the scikit-learn models which use highly optimized C libraries like LIBLINEAR.